In [1]:
import matplotlib
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
from upsetplot import UpSet, from_contents, plot, from_indicators
import numpy as np
import seaborn as sns 
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings("ignore")

In [2]:
def read_csv_data(mfile):
    adata = pd.read_csv(mfile,sep=",",header=0)
    adata.columns = ['gene','C','Z','Pval','FDR','cluster']
    adata = adata[adata['cluster'] > 0].copy()
    adata['cluster'] = adata['cluster'].astype(int)  # 转换为整数
    return adata

#筛选出指定的cluster（3.17修改）
def find_specfic_cluster(data,target_cluster):
    filterd_df = data[data['cluster'].isin(target_cluster)]#选择target_cluster的行
    return filterd_df

#生成每个细胞类型的字典
def module_gene_dic(data,name):
    indicators_df= from_contents({key:df['gene'].tolist() for key,df in data.items()})
    upset = UpSet(indicators_df,sort_by='degree',subset_size='count',show_counts=True,min_degree=(len(data) - 4))
    upset.plot()

    test = indicators_df.reset_index().set_index('id')#将索引重置，即将原来的索引列（默认是index）转换为普通列，并生成一个新的默认索引。
    sss = test.sum(axis =1).reset_index()#按行求和
    sss.columns = ['gene','count']
    result = sss.sort_values('count',ascending=False)
    result['type'] = name
    return result

def ancestral_input_data(data,name,save = None):
    counts = len(data)/4
    indicators_df= from_contents({key: list(set(df['OrthoID'].tolist())) for key,df in data.items()})
    df= indicators_df.reset_index()
    # df.columns = ['gene','type']
    df =df.replace({True: 1, False: 0})
    df['counts'] = df.drop('id', axis=1).sum(axis=1)
    # 按counts降序排列
    df = df.sort_values(by='counts', ascending=False).reset_index(drop=True)
    df = df[df['counts'] >= counts]
    df['counts'] = df['counts'].astype(str) + ' (' + name + ')'
    if save:
        df.to_csv(save, index=False)
    df
    return df


In [3]:
gene = pd.read_csv("D:/project/digestion_SI/04.CCM/ortholog_input.csv")

In [4]:
gene

,OrthoID,OrthoGroup,MaxGeneNum,alligator_gar,freshwater_butterflyfish,gray_bichir,human,indian_medaka,lampreyv2,lungfish,mississippi_paddlefish,tarpon,whitespotted_bambooshark,nwTree
0,OrthoGroup1,OG0000001_tree,2,KIF1C,KIF1C,KIF1C,KIF1C,KIF1C,NaN,NaN,KIF1C,MATL_G00070770;KIF1B,NaN,"((tarpon_pep_KIF1B:0.055528,tarpon_pep_MATL_G0..."
1,OrthoGroup2,OG0000001_tree,2,KIF15,KIF15,KIF15,KIF15;ENSG00000280610,KIF15,KIF15,KIF15,KIF15,NaN,KIF15,(((whitespotted_bambooshark_pep_KIF15:0.342944...
2,OrthoGroup3,OG0000001_tree,2,KIF17,KIF17,KIF17,KIF17,KIF17,lamprey10780,KIF17,KIF17;LOC121328603,KIF17,KIF17,"((((gray_bichir_pep_KIF17:0.206108,((alligator..."
3,OrthoGroup4,OG0000001_tree,2,KIF4A,evm.model.ptg000022l.210.1.65d20acf,KIF4A,KIF4A;KIF4B,KIF4A,lamprey24544,NaN,LOC121295238;KIF4A,MATL_G00053260;KIF4A,KIF4A,"(((((human_pep_human_pep_KIF4A:0.0162852,human..."
4,OrthoGroup5,OG0000001_tree,3,KIF3A,KIF3A,KIF3A,KIF3A,KIF3A,KIF3A,KIF3A,KIF3A,KIF3A,KIF3A,"(((((((((gray_bichir_pep_KIF3A:0.0334102,(huma..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12794,OrthoGroup12795,OG0014567_tree,1,NaN,NaN,NaN,RPUSD3,NaN,RPUSD3,NaN,RPUSD3,NaN,RPUSD3,"(lampreyv2_pep_RPUSD3:0.344625,(mississippi_pa..."
12795,OrthoGroup12796,OG0014568_tree,1,NaN,NaN,NaN,SERPINI2,NaN,NaN,SERPINI2,SERPINI2,NaN,SERPINI2,"(mississippi_paddlefish_pep_SERPINI2:0.156298,..."
12796,OrthoGroup12797,OG0014574_tree,1,NaN,NaN,NaN,RAD51AP2,LOC112158468,lamprey10664,RAD51AP2,NaN,NaN,NaN,"(lampreyv2_pep_lamprey10664:0.957639,(indian_m..."
12797,OrthoGroup12798,OG0014575_tree,1,NaN,NaN,NaN,IFNLR1,NaN,lamprey20082,IFNLR1,NaN,NaN,IFNLR1,"(lampreyv2_pep_lamprey20082:0.934728,(lungfish..."


In [5]:
csv_data_dic = {}
csv_data_dic['Ome.I'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/01.Oryzias_melastigma_intestine60.modules4.0.csv') 
csv_data_dic['Mat.I'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/02.Megalops_atlanticus_intestine60.modules4.0.csv') 
csv_data_dic['Psp.I'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/03.Polyodon_spathula_intestine70.modules4.0.csv' ) 
csv_data_dic['Pan.I'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/04.Protopterus_annectens_intestine60.modules4.0.csv')   
csv_data_dic['Pse.I'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/05.Polypterus_senegalus_intestine60.modules4.0.csv')
csv_data_dic['Cpl.I'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/06.Chiloscyllium_plagiosum_intestine60.modules4.0.csv') 
csv_data_dic['Lre.G'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/07.Lampetra_japonica_intestine40.modules4.0.csv')
csv_data_dic['Pbu.I'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/08.Pantodon_buchholzi_intestine80.modules4.0.csv')
csv_data_dic['Asp.I'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/09.Atractosteus_spatula_intestine60.modules4.0.csv')
csv_data_dic['Hsa.I'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/00.Homo_sapiens_intestine60.2.modules4.0.csv')

csv_data_dic['Ome.S'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/01.Oryzias_melastigma_stomach40.modules4.0.csv') 
csv_data_dic['Mat.S'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/02.Megalops_atlanticus_stomach60.2.modules4.0.csv') 
csv_data_dic['Psp.S'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/03.Polyodon_spathula_stomach60.modules4.0.csv') 
csv_data_dic['Pan.S'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/04.Protopterus_annectens_stomach60.modules4.0.csv')   
csv_data_dic['Pse.S'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/05.Polypterus_senegalus_stomach70.modules4.0.csv')
csv_data_dic['Cpl.S'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/06.Chiloscyllium_plagiosum_stomach60.modules4.0.csv') 
csv_data_dic['Pbu.S'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/08.Pantodon_buchholzi_stomach60.modules4.0.csv')
csv_data_dic['Asp.S'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/09.Atractosteus_spatula_stomach80.modules4.0.csv')
csv_data_dic['Hsa.S'] = read_csv_data('D:/project/digestion_SI/03.hotspot/modules2/00.Homo_sapiens_stomach60.2.modules4.0.csv') 

In [6]:
# 你的物种映射字典  
dataset_to_ortho_species = {    
    'Hsa.I': 'human',    
    'Hsa.S': 'human',    
    'Ome.I': 'indian_medaka',    
    'Ome.S': 'indian_medaka',    
    'Mat.I': 'tarpon',    
    'Mat.S': 'tarpon',    
    'Psp.I': 'mississippi_paddlefish',    
    'Psp.S': 'mississippi_paddlefish',    
    'Pan.I': 'lungfish',    
    'Pan.S': 'lungfish',    
    'Pse.I': 'gray_bichir',    
    'Pse.S': 'gray_bichir',    
    'Cpl.I': 'whitespotted_bambooshark',  
    'Cpl.S': 'whitespotted_bambooshark',    
    'Pbu.I': 'freshwater_butterflyfish',    
    'Pbu.S': 'freshwater_butterflyfish',    
    'Asp.I': 'alligator_gar',    
    'Asp.S': 'alligator_gar',    
    'Lre.G': 'lampreyv2',    
}  
  
# ===================== 核心修改：预先构建展开的映射字典 =====================  
# 结构: expanded_map[species_code] = { 单个基因名: OrthoID, ... }  
# 处理分号分隔的多基因情况  
  
expanded_map = {}  # {species_name: {gene_symbol: OrthoID}}  
  
for species_code in set(dataset_to_ortho_species.values()):  
    expanded_map[species_code] = {}  
  
for _, row in gene.iterrows():  
    ortho_id = row['OrthoID']  
      
    for species_code in set(dataset_to_ortho_species.values()):  
        if species_code not in gene.columns:  
            continue  
          
        cell = row[species_code]  
          
        # 跳过 NaN  
        if pd.isna(cell):  
            continue  
          
        # 将 cell 转为字符串，按分号拆分，逐个基因建立映射  
        cell_str = str(cell).strip()  
        if cell_str == '' or cell_str.lower() == 'nan':  
            continue  
          
        individual_genes = [g.strip() for g in cell_str.split(';') if g.strip()]  
          
        for single_gene in individual_genes:  
            # 如果同一个基因出现在多个 OrthoGroup 中，后者会覆盖前者  
            # 可以根据需求决定是否保留第一个  
            if single_gene not in expanded_map[species_code]:  
                expanded_map[species_code][single_gene] = ortho_id  
            else:  
                # 如果你想看冲突，可以取消下面的注释  
                # print(f"⚠️ 冲突: {species_code} 中基因 {single_gene} "  
                #       f"已映射到 {expanded_map[species_code][single_gene]}，"  
                #       f"新值 {ortho_id} 被忽略")  
                pass  
  
print("✅ 展开映射字典构建完成！")  
for sp, mp in expanded_map.items():  
    print(f"   {sp}: 共 {len(mp)} 个基因 → OrthoID 映射")  
  
# ===================== 遍历处理每个数据集 =====================  
for key, df in csv_data_dic.items():  
      
    # 1. 从映射字典中获取对应的 ortho 物种名  
    if key not in dataset_to_ortho_species:  
        print(f"⚠️ 警告：{key} 不在映射字典中，跳过")  
        continue  
      
    species_code = dataset_to_ortho_species[key]  
      
    # 2. 检查这个物种是否有展开映射  
    if species_code not in expanded_map or len(expanded_map[species_code]) == 0:  
        print(f"⚠️ 警告：{species_code} 没有可用的基因映射，跳过")  
        continue  
      
    # 3. 用展开后的映射字典进行映射  
    gene2ortho = expanded_map[species_code]  
      
    # 4. 新增 OrthoID 列  
    df['OrthoID'] = df['gene'].map(gene2ortho)  
      
    # 5. 统计匹配情况  
    matched = df['OrthoID'].notna().sum()  
    total = len(df)  
      
    # 6. 没匹配到则保留原基因名  
    df['OrthoID'] = df['OrthoID'].fillna(df['gene'])  
      
    # 放回字典  
    csv_data_dic[key] = df  
      
    print(f"   {key} ({species_code}): {matched}/{total} 个基因成功映射到 OrthoID")  
  
print("\n✅ 全部处理完成！所有表已新增 OrthoID 列")  


✅ 展开映射字典构建完成！
   indian_medaka: 共 17054 个基因 → OrthoID 映射
   alligator_gar: 共 14662 个基因 → OrthoID 映射
   freshwater_butterflyfish: 共 16717 个基因 → OrthoID 映射
   mississippi_paddlefish: 共 24392 个基因 → OrthoID 映射
   human: 共 15903 个基因 → OrthoID 映射
   lampreyv2: 共 7742 个基因 → OrthoID 映射
   whitespotted_bambooshark: 共 14627 个基因 → OrthoID 映射
   gray_bichir: 共 14936 个基因 → OrthoID 映射
   lungfish: 共 14888 个基因 → OrthoID 映射
   tarpon: 共 18619 个基因 → OrthoID 映射
   Ome.I (indian_medaka): 5163/5647 个基因成功映射到 OrthoID
   Mat.I (tarpon): 4386/4765 个基因成功映射到 OrthoID
   Psp.I (mississippi_paddlefish): 3974/4295 个基因成功映射到 OrthoID
   Pan.I (lungfish): 3813/4157 个基因成功映射到 OrthoID
   Pse.I (gray_bichir): 4558/4871 个基因成功映射到 OrthoID
   Cpl.I (whitespotted_bambooshark): 3807/4141 个基因成功映射到 OrthoID
   Lre.G (lampreyv2): 2582/3668 个基因成功映射到 OrthoID
   Pbu.I (freshwater_butterflyfish): 5023/6130 个基因成功映射到 OrthoID
   Asp.I (alligator_gar): 5182/5531 个基因成功映射到 OrthoID
   Hsa.I (human): 5420/5816 个基因成功映射到 OrthoID
   Ome.S (indian_

In [7]:
#Endothelial
Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[5])
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[3])
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[4])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[4])
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[14,3])
Cpl_I = find_specfic_cluster(csv_data_dic['Cpl.I'],[3])
Lre_G = find_specfic_cluster(csv_data_dic['Lre.G'],[4])
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[4])
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[3])
Hsa_I_L = find_specfic_cluster(csv_data_dic['Hsa.I'],[32,3])
Hsa_I_V = find_specfic_cluster(csv_data_dic['Hsa.I'],[3,25,33,24,22,28])

Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[6])
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[8])
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[5])
Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[5])
Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[2])
Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[4])
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[6,14])
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[5])
Hsa_S_L = find_specfic_cluster(csv_data_dic['Hsa.S'],[2,28,15,16,25,9,18,22,11,13,19])
Hsa_S_V = find_specfic_cluster(csv_data_dic['Hsa.S'],[28,2,13])

Endothelial = {
    'Ome_I.Endothelial':Ome_I,
    'Mat_I.Endothelial':Mat_I,
    'Psp_I.Endothelial':Psp_I,
    'Pan_I.Endothelial':Pan_I,
    'Pse_I.Endothelial':Pse_I,
    'Cpl_I.Endothelial':Cpl_I,
    'Lre_G.Endothelial':Lre_G,
    'Pbu_I.Endothelial':Pbu_I,
    'Asp_I.Endothelial':Asp_I,
    'Hsa_I.Lymphatic_endothelia':Hsa_I_L,
    'Hsa_I.Vascular_endothelia':Hsa_I_V,
    
    'Ome_D.Endothelial':Ome_S,
    'Mat_S.Endothelial':Mat_S,
    'Psp_S.Endothelial':Psp_S,
    'Pan_D.Endothelial':Pan_S,
    'Pse_S.Endothelial':Pse_S,
    'Cpl_S.Endothelial':Cpl_S,
    'Pbu_S.Endothelial':Pbu_S,
    'Asp_S.Endothelial':Asp_S, 
    'Hsa_S.Lymphatic_endothelia':Hsa_S_L,
    'Hsa_S.Vascular_endothelia':Hsa_S_V,
}

In [8]:
Endothelial_ai = ancestral_input_data(Endothelial,'Endothelial')
Endothelial_ai

,Ome_I.Endothelial,Mat_I.Endothelial,Psp_I.Endothelial,Pan_I.Endothelial,Pse_I.Endothelial,Cpl_I.Endothelial,Lre_G.Endothelial,Pbu_I.Endothelial,Asp_I.Endothelial,Hsa_I.Lymphatic_endothelia,...,Psp_S.Endothelial,Pan_D.Endothelial,Pse_S.Endothelial,Cpl_S.Endothelial,Pbu_S.Endothelial,Asp_S.Endothelial,Hsa_S.Lymphatic_endothelia,Hsa_S.Vascular_endothelia,id,counts
0,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup1007,21 (Endothelial)
1,1,1,1,1,1,1,0,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup2117,20 (Endothelial)
2,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,1,1,1,OrthoGroup8386,20 (Endothelial)
3,1,1,1,1,1,1,0,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup6084,20 (Endothelial)
4,1,1,1,1,1,1,0,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup305,20 (Endothelial)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,1,1,OrthoGroup2132,6 (Endothelial)
409,1,0,1,0,1,0,0,0,0,0,...,1,0,0,1,0,0,0,0,OrthoGroup2068,6 (Endothelial)
410,0,1,1,0,0,0,0,1,0,0,...,0,0,1,0,1,0,0,0,OrthoGroup1308,6 (Endothelial)
411,1,0,0,0,1,1,0,0,0,0,...,0,0,1,1,0,0,0,0,OrthoGroup3136,6 (Endothelial)


In [9]:
#Endocrine
Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[15,2]) #,13,20,16,8,17
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[12,20])#,2,9,14,10,18,7,27,28,22
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[9])#11,19,8
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[14])#,5,2,1,21,
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[9])#2,6,4,3,10,25,18,22,19
Cpl_I = find_specfic_cluster(csv_data_dic['Cpl.I'],[12])#,15,17,10
Lre_G = find_specfic_cluster(csv_data_dic['Lre.G'],[6,16])#,2,3,18,14,12
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[8,15,16])#,1,21,22
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[8])#6,16,17,12,25,18,27,32
Hsa_I = find_specfic_cluster(csv_data_dic['Hsa.I'],[36])#,18,23,19,34

Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[7,8,12])#,24,6,27,29
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[16,28])#
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[6])#,18,23,15,12,7,13,25,21
Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[15])#,12,23,16,14
Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[4])#4,9,11,10,17,21,19,23,8,5,22,15,24
Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[9])#1,13,18,6,19
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[9,25])#,16,2,20,
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[6])#2,8,10,14,5
Hsa_S = find_specfic_cluster(csv_data_dic['Hsa.S'],[20])#,4,27,9,19,13,3,21

Endocrine = {
    'Ome_I.Enteroendocrine':Ome_I,
    'Mat_I.Enteroendocrine':Mat_I,
    'Psp_I.Enteroendocrine':Psp_I,
    'Pan_I.Enteroendocrine':Pan_I,
    'Pse_I.Enteroendocrine':Pse_I,
    'Cpl_I.Enteroendocrine':Cpl_I,
    'Lre_G.Enteroendocrine':Lre_G,
    'Pbu_I.Enteroendocrine':Pbu_I,
    'Asp_I.Enteroendocrine':Asp_I,
    'Hsa_I.Enteroendocrine':Hsa_I,
    
    'Ome_D.Enteroendocrine':Ome_S,
    'Mat_S.Enteroendocrine':Mat_S,
    'Psp_S.Enteroendocrine':Psp_S,
    'Pan_D.Enteroendocrine':Pan_S,
    'Pse_S.Enteroendocrine':Pse_S,
    'Cpl_S.Enteroendocrine':Cpl_S,
    'Pbu_S.Enteroendocrine':Pbu_S,
    'Asp_S.Enteroendocrine':Asp_S, 
    'Hsa_S.Enteroendocrine':Hsa_S,
}

In [10]:
Endocrine_ai = ancestral_input_data(Endocrine,'Endocrine',save='D:/project/digestion_SI/04.CCM/Endocrine_20260405.csv')
Endocrine_ai

,Ome_I.Enteroendocrine,Mat_I.Enteroendocrine,Psp_I.Enteroendocrine,Pan_I.Enteroendocrine,Pse_I.Enteroendocrine,Cpl_I.Enteroendocrine,Lre_G.Enteroendocrine,Pbu_I.Enteroendocrine,Asp_I.Enteroendocrine,Hsa_I.Enteroendocrine,...,Mat_S.Enteroendocrine,Psp_S.Enteroendocrine,Pan_D.Enteroendocrine,Pse_S.Enteroendocrine,Cpl_S.Enteroendocrine,Pbu_S.Enteroendocrine,Asp_S.Enteroendocrine,Hsa_S.Enteroendocrine,id,counts
0,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup3070,19 (Endocrine)
1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup1154,19 (Endocrine)
2,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup326,19 (Endocrine)
3,1,1,1,1,1,1,0,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup1278,18 (Endocrine)
4,1,1,1,1,1,1,0,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup7049,18 (Endocrine)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,0,0,0,0,0,0,0,1,1,0,...,0,0,0,1,0,1,1,0,OrthoGroup4630,5 (Endocrine)
290,0,1,0,0,0,1,0,0,1,0,...,0,0,0,0,0,1,0,1,OrthoGroup4651,5 (Endocrine)
291,1,1,1,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,OrthoGroup2440,5 (Endocrine)
292,0,0,0,0,1,1,0,1,1,0,...,0,0,0,0,1,0,0,0,FAM20C,5 (Endocrine)


In [11]:
#Myeloid
#新的module
Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[9,14])
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[15,6,8])
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[5])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[16,20])
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[5,8])
Cpl_I = find_specfic_cluster(csv_data_dic['Cpl.I'],[5,7])
Lre_G = find_specfic_cluster(csv_data_dic['Lre.G'],[5,9])
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[2,7])
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[2,10])
Hsa_I = find_specfic_cluster(csv_data_dic['Hsa.I'],[9])

Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[4,11,13])
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[3,5])
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[3])
Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[6])
Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[12,14])
Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[10])
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[18,4])
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[7,4,9])
Hsa_S = find_specfic_cluster(csv_data_dic['Hsa.S'],[8,23])	

Myeloid = {
    'Ome_I.Myeloid':Ome_I,
    'Mat_I.Myeloid':Mat_I,
    'Psp_I.Myeloid':Psp_I,
    'Pan_I.Myeloid':Pan_I,
    'Pse_I.Myeloid':Pse_I,
    'Cpl_I.Myeloid':Cpl_I,
    'Lre_G.Myeloid':Lre_G,
    'Pbu_I.Myeloid':Pbu_I,
    'Asp_I.Myeloid':Asp_I,
    'Hsa_I.Myeloid':Hsa_I,
    
    'Ome_D.Myeloid':Ome_S,
    'Mat_S.Myeloid':Mat_S,
    'Psp_S.Myeloid':Psp_S,
    'Pan_D.Myeloid':Pan_S,
    'Pse_S.Myeloid':Pse_S,
    'Cpl_S.Myeloid':Cpl_S,
    'Pbu_S.Myeloid':Pbu_S,
    'Asp_S.Myeloid':Asp_S, 
    'Hsa_S.Myeloid':Hsa_S,
}

In [12]:
Myeloid_ai = ancestral_input_data(Myeloid,'Myeloid')
Myeloid_ai

,Ome_I.Myeloid,Mat_I.Myeloid,Psp_I.Myeloid,Pan_I.Myeloid,Pse_I.Myeloid,Cpl_I.Myeloid,Lre_G.Myeloid,Pbu_I.Myeloid,Asp_I.Myeloid,Hsa_I.Myeloid,...,Mat_S.Myeloid,Psp_S.Myeloid,Pan_D.Myeloid,Pse_S.Myeloid,Cpl_S.Myeloid,Pbu_S.Myeloid,Asp_S.Myeloid,Hsa_S.Myeloid,id,counts
0,1,1,1,1,1,1,1,1,1,1,...,1,0,1,1,1,1,1,1,OrthoGroup354,18 (Myeloid)
1,1,1,1,1,1,0,1,1,1,1,...,1,1,1,1,0,1,1,1,OrthoGroup3049,17 (Myeloid)
2,1,1,1,1,1,1,0,0,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup8084,17 (Myeloid)
3,1,1,1,1,1,1,0,1,0,1,...,1,1,1,1,1,1,0,1,OrthoGroup6750,16 (Myeloid)
4,1,1,1,1,1,1,0,1,1,1,...,1,1,1,1,1,0,1,0,OrthoGroup4978,16 (Myeloid)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
406,0,1,0,1,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,CLEC17A,5 (Myeloid)
407,1,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,1,OrthoGroup1959,5 (Myeloid)
408,1,0,0,0,1,0,0,1,0,0,...,0,0,0,1,0,1,0,0,OrthoGroup9563,5 (Myeloid)
409,0,1,0,0,0,1,0,0,1,0,...,0,0,0,1,0,0,0,1,OrthoGroup5482,5 (Myeloid)


In [13]:
#祖先溯源版
Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[3])
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[5])
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[15])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[10])
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[11])
Cpl_I = find_specfic_cluster(csv_data_dic['Cpl.I'],[8])
Lre_G = find_specfic_cluster(csv_data_dic['Lre.G'],[20,13,8])#1,20,11,8
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[9])
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[9,11])
Hsa_I = find_specfic_cluster(csv_data_dic['Hsa.I'],[6,15])

Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[19])
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[14])#1,12,14
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[11])
Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[8])
Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[6])
Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[8])
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[8])
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[15,11])
Hsa_S = find_specfic_cluster(csv_data_dic['Hsa.S'],[7])

T_cell = {
    'Ome_I.T_cells':Ome_I,
    'Mat_I.T_cells':Mat_I,
    'Psp_I.T_cells':Psp_I,
    'Pan_I.T_cells':Pan_I,
    'Pse_I.T_cells':Pse_I,
    'Cpl_I.T_cells':Cpl_I,
    'Lre_G.T_like':Lre_G,
    'Pbu_I.T_cells':Pbu_I,
    'Asp_I.T_cells':Asp_I,
    'Hsa_I.T_NK':Hsa_I,
    
    'Ome_D.T_cells':Ome_S,
    'Mat_S.T_cells':Mat_S,
    'Psp_S.T_cells':Psp_S,
    'Pan_D.T_cells':Pan_S,
    'Pse_S.T_cells':Pse_S,
    'Cpl_S.T_cells':Cpl_S,
    'Pbu_S.T_cells':Pbu_S,
    'Asp_S.T_cells':Asp_S, 
    'Hsa_S.T_NK':Hsa_S,
}

In [14]:
T_cell_ai = ancestral_input_data(T_cell,'T_cell')
T_cell_ai

,Ome_I.T_cells,Mat_I.T_cells,Psp_I.T_cells,Pan_I.T_cells,Pse_I.T_cells,Cpl_I.T_cells,Lre_G.T_like,Pbu_I.T_cells,Asp_I.T_cells,Hsa_I.T_NK,...,Mat_S.T_cells,Psp_S.T_cells,Pan_D.T_cells,Pse_S.T_cells,Cpl_S.T_cells,Pbu_S.T_cells,Asp_S.T_cells,Hsa_S.T_NK,id,counts
0,1,1,1,1,1,1,0,1,1,1,...,1,1,1,1,1,1,1,1,OrthoGroup139,18 (T_cell)
1,1,1,1,1,1,1,0,1,1,1,...,0,1,1,1,1,1,1,1,OrthoGroup11648,17 (T_cell)
2,1,1,1,0,1,1,1,1,1,1,...,1,1,1,0,1,1,1,1,OrthoGroup1289,17 (T_cell)
3,1,1,1,1,1,1,0,1,1,1,...,1,0,1,1,1,1,1,1,OrthoGroup7848,16 (T_cell)
4,1,1,1,0,1,1,0,1,1,1,...,1,1,1,1,0,1,1,1,OrthoGroup11188,15 (T_cell)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218,1,0,0,0,0,1,0,0,0,1,...,0,0,1,0,1,0,0,0,OrthoGroup5642,5 (T_cell)
219,1,0,0,0,0,1,0,1,1,0,...,0,0,0,0,0,0,1,0,OrthoGroup1649,5 (T_cell)
220,1,1,0,0,0,0,0,0,0,0,...,0,0,1,1,0,1,0,0,OrthoGroup1392,5 (T_cell)
221,1,1,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,OrthoGroup3608,5 (T_cell)


In [15]:
#Mesenchymal
Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[6,7,12,19])
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[4])
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[10,13])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[6,11])
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[4,3,14])
Cpl_I = find_specfic_cluster(csv_data_dic['Cpl.I'],[2])
Lre_G = find_specfic_cluster(csv_data_dic['Lre.G'],[22,1,4,12])
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[5])
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[19,3])

Hsa_I_SMC = find_specfic_cluster(csv_data_dic['Hsa.I'],[14,34,30,16,31])
Hsa_I_L = find_specfic_cluster(csv_data_dic['Hsa.I'],[29,35,26])
Hsa_I_P = find_specfic_cluster(csv_data_dic['Hsa.I'],[10,26])
Hsa_I_F = find_specfic_cluster(csv_data_dic['Hsa.I'],[2])
Hsa_I_MF = find_specfic_cluster(csv_data_dic['Hsa.I'],[26])

Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[9,17])
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[13,15])
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[8])
Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[2])
Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[3])
Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[3])
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[7,11])
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[3,1])

Hsa_S_SMC = find_specfic_cluster(csv_data_dic['Hsa.S'],[28]) #?
Hsa_S_L = find_specfic_cluster(csv_data_dic['Hsa.S'],[27,15])
Hsa_S_P = find_specfic_cluster(csv_data_dic['Hsa.S'],[17])
Hsa_S_F = find_specfic_cluster(csv_data_dic['Hsa.S'],[4,28])
Hsa_S_MF = find_specfic_cluster(csv_data_dic['Hsa.S'],[4,28])

Mesenchymal = {
    'Ome_I.Mesenchymal':Ome_I,
    'Mat_I.Mesenchymal':Mat_I,
    'Psp_I.Mesenchymal':Psp_I,
    'Pan_I.Mesenchymal':Pan_I,
    'Pse_I.Mesenchymal':Pse_I,
    'Cpl_I.Mesenchymal':Cpl_I,
    'Lre_G.Mesenchymal':Lre_G,
    'Pbu_I.Mesenchymal':Pbu_I,
    'Asp_I.Mesenchymal':Asp_I,

    'Hsa_I.Smooth_muscle':Hsa_I_SMC,
    'Hsa_I.Lymphoid_stromal':Hsa_I_L,
    'Hsa_I.Fibroblast':Hsa_I_F,
    'Hsa_I.Pericyte':Hsa_I_P,
    'Hsa_I.Myofibroblast':Hsa_I_MF,
    
    'Ome_D.Mesenchymal':Ome_S,
    'Mat_S.Mesenchymal':Mat_S,
    'Psp_S.Mesenchymal':Psp_S,
    'Pan_D.Mesenchymal':Pan_S,
    'Pse_S.Mesenchymal':Pse_S,
    'Cpl_S.Mesenchymal':Cpl_S,
    'Pbu_S.Mesenchymal':Pbu_S,
    'Asp_S.Mesenchymal':Asp_S,

    'Hsa_S.Smooth_muscle':Hsa_S_SMC,
    'Hsa_S.Lymphoid_stromal':Hsa_S_L,
    'Hsa_S.Fibroblast':Hsa_S_F,
    'Hsa_S.Pericyte':Hsa_S_P,
    'Hsa_S.Myofibroblast':Hsa_S_MF,
}

In [16]:
Mesenchymal_ai = ancestral_input_data(Mesenchymal,'Mesenchymal')
Mesenchymal_ai

,Ome_I.Mesenchymal,Mat_I.Mesenchymal,Psp_I.Mesenchymal,Pan_I.Mesenchymal,Pse_I.Mesenchymal,Cpl_I.Mesenchymal,Lre_G.Mesenchymal,Pbu_I.Mesenchymal,Asp_I.Mesenchymal,Hsa_I.Smooth_muscle,...,Cpl_S.Mesenchymal,Pbu_S.Mesenchymal,Asp_S.Mesenchymal,Hsa_S.Smooth_muscle,Hsa_S.Lymphoid_stromal,Hsa_S.Fibroblast,Hsa_S.Pericyte,Hsa_S.Myofibroblast,id,counts
0,1,1,1,1,1,1,1,1,1,0,...,1,1,1,0,0,1,0,1,OrthoGroup8554,20 (Mesenchymal)
1,1,1,1,1,1,1,0,1,1,0,...,1,1,1,0,0,1,0,1,OrthoGroup650,19 (Mesenchymal)
2,1,1,1,0,1,1,0,1,1,1,...,1,1,1,0,1,1,0,1,OrthoGroup1561,19 (Mesenchymal)
3,0,1,1,1,1,1,1,1,1,0,...,1,1,1,0,0,1,0,1,FBN1,18 (Mesenchymal)
4,1,1,1,1,1,1,0,1,1,0,...,1,1,1,0,0,1,0,1,OrthoGroup3879,18 (Mesenchymal)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
322,1,0,1,0,1,1,0,0,0,0,...,1,0,1,0,0,0,0,0,OrthoGroup4748,7 (Mesenchymal)
323,0,0,1,0,0,1,0,0,0,0,...,1,0,0,0,0,1,0,1,OrthoGroup2844,7 (Mesenchymal)
324,0,0,1,0,0,0,0,1,0,1,...,0,0,0,0,0,1,0,1,OrthoGroup681,7 (Mesenchymal)
325,1,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,1,0,1,OrthoGroup8800,7 (Mesenchymal)


In [17]:
#Enterocyte

Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[4])
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[1,7])
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[3])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[1,21])
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[6])
Cpl_I = find_specfic_cluster(csv_data_dic['Cpl.I'],[4])
Lre_G = find_specfic_cluster(csv_data_dic['Lre.G'],[17,12])
Lre_G_like = find_specfic_cluster(csv_data_dic['Lre.G'],[3])
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[6,18])
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[1,4,15])
Hsa_I = find_specfic_cluster(csv_data_dic['Hsa.I'],[12,11,21])
Hsa_I_M = find_specfic_cluster(csv_data_dic['Hsa.I'],[5,4,26])

Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[3,14])
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[1,19])
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[20,21])
Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[14,17])
# Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[])
# Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[])
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[1,2])
# Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[])
Hsa_S = find_specfic_cluster(csv_data_dic['Hsa.S'],[6,12])

Enterocyte = {
    'Ome_I.Enterocytes':Ome_I,
    'Mat_I.Enterocytes':Mat_I,
    'Psp_I.Enterocytes':Psp_I,
    'Pan_I.Enterocytes':Pan_I,
    'Pse_I.Enterocytes':Pse_I,
    'Cpl_I.Enterocytes':Cpl_I,
    'Lre_G.Enterocytes':Lre_G,
    'Lre_G.Enterocytes_like':Lre_G_like,
    'Pbu_I.Enterocytes':Pbu_I,
    'Asp_I.Enterocytes':Asp_I,
    'Hsa_I.Enterocytes':Hsa_I,
    'Hsa_I.Microfold':Hsa_I_M,
    
    'Ome_D.Enterocytes':Ome_S,
    'Mat_S.Enterocytes':Mat_S,
    'Psp_S.Enterocytes':Psp_S,
    'Pan_D.Enterocytes':Pan_S,
    # 'Pse_S.':Pse_S,
    # 'Cpl_S.':Cpl_S,
    'Pbu_S.Enterocytes':Pbu_S,
    # 'Asp_S.':Asp_S, 
    'Hsa_S.Enterocytes':Hsa_S,
}

In [18]:
Enterocyte_ai = ancestral_input_data(Enterocyte,'Enterocyte')
Enterocyte_ai

,Ome_I.Enterocytes,Mat_I.Enterocytes,Psp_I.Enterocytes,Pan_I.Enterocytes,Pse_I.Enterocytes,Cpl_I.Enterocytes,Lre_G.Enterocytes,Lre_G.Enterocytes_like,Pbu_I.Enterocytes,Asp_I.Enterocytes,Hsa_I.Enterocytes,Hsa_I.Microfold,Ome_D.Enterocytes,Mat_S.Enterocytes,Psp_S.Enterocytes,Pan_D.Enterocytes,Pbu_S.Enterocytes,Hsa_S.Enterocytes,id,counts
0,1,1,1,1,1,1,0,1,1,1,0,1,1,1,1,1,1,1,OrthoGroup2628,16 (Enterocyte)
1,1,1,1,1,1,1,0,0,1,1,0,1,1,1,1,1,1,1,OrthoGroup5637,15 (Enterocyte)
2,1,1,1,1,1,1,0,1,1,1,0,1,1,0,1,1,1,1,MGAM,15 (Enterocyte)
3,1,1,1,1,0,1,0,1,1,1,1,0,1,1,1,1,1,1,OrthoGroup9451,15 (Enterocyte)
4,1,1,1,1,1,0,0,1,1,1,0,1,1,1,1,1,1,1,OrthoGroup3617,15 (Enterocyte)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,0,0,1,1,1,0,0,0,0,1,0,1,0,0,0,0,0,0,OrthoGroup3942,5 (Enterocyte)
326,0,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,OrthoGroup149,5 (Enterocyte)
327,0,1,1,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,OrthoGroup2309,5 (Enterocyte)
328,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,OrthoGroup2690,5 (Enterocyte)


In [19]:
#B_cell
#New module
# # Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[])
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[26,1])#?
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[1,12])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[8,7,19,12])
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[16])
Cpl_I = find_specfic_cluster(csv_data_dic['Cpl.I'],[1,25])
Lre_G = find_specfic_cluster(csv_data_dic['Lre.G'],[1])
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[21])#?
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[25,29])
Hsa_I = find_specfic_cluster(csv_data_dic['Hsa.I'],[8,17])	

# # Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[])
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[1])#?
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[15])
# # Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[])
Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[1,16,8,19])
Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[2])#?
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[17])
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[17])
Hsa_S = find_specfic_cluster(csv_data_dic['Hsa.S'],[26])

B_cell = {
    # 'Ome_I.':Ome_I,
    'Mat_I.B_cells':Mat_I,
    'Psp_I.B_cells':Psp_I,
    'Pan_I.B_cells':Pan_I,
    'Pse_I.B_cells':Pse_I,
    'Cpl_I.B_cells':Cpl_I,
    'Lre_G.B_like':Lre_G,
    'Pbu_I.B_cells':Pbu_I,
    'Asp_I.B_cells':Asp_I,
    'Hsa_I.B_plasma':Hsa_I,
    
    # 'Ome_D.':Ome_S,
    'Mat_S.B_cells':Mat_S,
    'Psp_S.B_cells':Psp_S,
    # 'Pan_D.':Pan_S,
    'Pse_S.B_cells':Pse_S,
    'Cpl_S.B_cells':Cpl_S,
    'Pbu_S.B_cells':Pbu_S,
    'Asp_S.B_cells':Asp_S, 
    'Hsa_S.B_plasma':Hsa_S,
}

In [20]:
B_cell_ai = ancestral_input_data(B_cell,'B_cell')
B_cell_ai

,Mat_I.B_cells,Psp_I.B_cells,Pan_I.B_cells,Pse_I.B_cells,Cpl_I.B_cells,Lre_G.B_like,Pbu_I.B_cells,Asp_I.B_cells,Hsa_I.B_plasma,Mat_S.B_cells,Psp_S.B_cells,Pse_S.B_cells,Cpl_S.B_cells,Pbu_S.B_cells,Asp_S.B_cells,Hsa_S.B_plasma,id,counts
0,0,1,0,1,1,0,0,1,1,0,1,1,0,1,0,1,OrthoGroup12568,9 (B_cell)
1,0,0,1,1,0,0,0,1,1,0,1,1,0,1,1,1,OrthoGroup12433,9 (B_cell)
2,0,1,1,1,0,0,0,1,1,0,1,1,0,1,1,0,OrthoGroup4996,9 (B_cell)
3,0,1,1,1,0,0,0,1,1,0,1,1,0,1,0,0,OrthoGroup7699,8 (B_cell)
4,0,1,0,1,0,0,0,1,1,0,1,1,0,0,1,1,OrthoGroup3549,8 (B_cell)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,1,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,OrthoGroup8727,4 (B_cell)
194,0,0,0,0,0,0,0,1,1,0,1,0,0,0,1,0,OrthoGroup8590,4 (B_cell)
195,0,0,0,0,0,0,0,1,1,0,1,0,0,0,1,0,OrthoGroup3647,4 (B_cell)
196,0,0,0,0,0,0,0,1,0,0,1,0,0,1,1,0,OrthoGroup5757,4 (B_cell)


In [21]:
#Mucus_Goblet
Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[17])
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[2])
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[7,8,21])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[5,13])
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[12,19])
Cpl_I_Glike = find_specfic_cluster(csv_data_dic['Cpl.I'],[22,17,6,13,21])
Cpl_I_G = find_specfic_cluster(csv_data_dic['Cpl.I'],[10])
Lre_G = find_specfic_cluster(csv_data_dic['Lre.G'],[18,14])
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[11])
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[12])
Hsa_I_G = find_specfic_cluster(csv_data_dic['Hsa.I'],[23])
Hsa_I_P = find_specfic_cluster(csv_data_dic['Hsa.I'],[34])#?
Hsa_I_F = find_specfic_cluster(csv_data_dic['Hsa.I'],[12,27])#?			

Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[29])
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[2,7,9,12,17,31])
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[7])
Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[7,9])
Pse_S_F = find_specfic_cluster(csv_data_dic['Pse.S'],[22]) #23 Feveolar
Pse_S_M = find_specfic_cluster(csv_data_dic['Pse.S'],[22])
Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[23,21,6,15,11])
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[15,22])
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[8])
Hsa_S_G = find_specfic_cluster(csv_data_dic['Hsa.S'],[10,21])#11,12,5,18
Hsa_S_F = find_specfic_cluster(csv_data_dic['Hsa.S'],[10,13])#11,12,5,18
# Hsa_S_M = find_specfic_cluster(csv_data_dic['Hsa.S'],[21,10,1])#11,12,5,18

Mucus_Goblet = {
    'Ome_I.Goblet':Ome_I,
    'Mat_I.Goblet':Mat_I,
    'Psp_I.Goblet':Psp_I,
    'Pan_I.Goblet':Pan_I,
    'Pse_I.Goblet':Pse_I,
    'Cpl_I.Goblet_like':Cpl_I_Glike,
    'Cpl_I.Goblet':Cpl_I_G,
    'Lre_G.Goblet':Lre_G,
    'Pbu_I.Goblet':Pbu_I,
    'Asp_I.Goblet':Asp_I,
    'Hsa_I.Goblet':Hsa_I_G,
    'Hsa_I.Paneth':Hsa_I_P,
    'Hsa_I.Foveolar':Hsa_I_F,
    
    'Ome_D.Mucus':Ome_S,
    'Mat_S.Mucus':Mat_S,
    'Psp_S.Mucus':Psp_S,
    'Pan_D.Mucus':Pan_S,

    'Pse_S.Foveolar':Pse_S_F,
    'Pse_S.Mucus':Pse_S_M,

    'Cpl_S.Mucus':Cpl_S,
    'Pbu_S.Mucus':Pbu_S,
    'Asp_S.Mucus':Asp_S, 
    'Hsa_S.Foveolar':Hsa_S_F,
    'Hsa_S.Goblet':Hsa_S_G,
    # 'Hsa_S.Paneth':Hsa_S_M,
}

In [22]:
Mucus_Goblet_ai = ancestral_input_data(Mucus_Goblet,'Mucus_Goblet')
Mucus_Goblet_ai

,Ome_I.Goblet,Mat_I.Goblet,Psp_I.Goblet,Pan_I.Goblet,Pse_I.Goblet,Cpl_I.Goblet_like,Cpl_I.Goblet,Lre_G.Goblet,Pbu_I.Goblet,Asp_I.Goblet,...,Pan_D.Mucus,Pse_S.Foveolar,Pse_S.Mucus,Cpl_S.Mucus,Pbu_S.Mucus,Asp_S.Mucus,Hsa_S.Foveolar,Hsa_S.Goblet,id,counts
0,1,1,1,1,0,0,1,0,1,1,...,1,1,1,1,0,1,1,1,OrthoGroup3434,17 (Mucus_Goblet)
1,1,1,1,1,1,0,1,0,1,1,...,1,1,1,1,1,0,0,1,OrthoGroup6272,16 (Mucus_Goblet)
2,1,1,1,1,1,0,1,0,0,1,...,1,1,1,1,0,0,1,1,OrthoGroup8421,16 (Mucus_Goblet)
3,1,1,1,1,0,0,1,0,1,1,...,1,1,1,1,0,0,1,1,OrthoGroup4056,16 (Mucus_Goblet)
4,1,1,1,0,0,0,1,0,1,1,...,0,1,1,0,1,1,1,1,OrthoGroup386,15 (Mucus_Goblet)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,1,1,0,1,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,OrthoGroup6294,6 (Mucus_Goblet)
136,0,0,1,0,1,0,1,0,0,1,...,0,0,0,1,0,0,0,1,OrthoGroup6382,6 (Mucus_Goblet)
137,0,1,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,OrthoGroup5652,6 (Mucus_Goblet)
138,0,0,1,0,0,0,1,0,1,0,...,0,0,0,1,1,0,0,1,OrthoGroup5989,6 (Mucus_Goblet)


In [23]:
#Ciliated
# Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[])
# Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[])
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[2])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[2])
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[2])
# Cpl_I = find_specfic_cluster(csv_data_dic['Cpl.I'],[])
Lre_G = find_specfic_cluster(csv_data_dic['Lre.G'],[2])
# Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[])
# Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[])
# Hsa_I = find_specfic_cluster(csv_data_dic['Hsa.I'],[])

# Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[])
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[4])
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[1])
Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[1,4])
Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[5])
Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[1])
# Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[])
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[2,6])
# Hsa_S = find_specfic_cluster(csv_data_dic['Hsa.S'],[])

Ciliated = {
    # 'Ome_I.':Ome_I,
    # 'Mat_I.':Mat_I,
    'Psp_I.Ciliated':Psp_I,
    'Pan_I.Ciliated':Pan_I,
    'Pse_I.Ciliated':Pse_I,
    # 'Cpl_I.':Cpl_I,
    'Lre_G.Ciliated':Lre_G,
    # 'Pbu_I.':Pbu_I,
    # 'Asp_I.':Asp_I,
    # 'Hsa_I.':Hsa_I,
    
    # 'Ome_D.':Ome_S,
    'Mat_S.Ciliated':Mat_S,
    'Psp_S.Ciliated':Psp_S,
    'Pan_D.Ciliated':Pan_S,
    'Pse_S.Ciliated':Pse_S,
    'Cpl_S.Ciliated':Cpl_S,
    # 'Pbu_S.':Pbu_S,
    'Asp_S.Ciliated':Asp_S, 
    # 'Hsa_S.':Hsa_S,
}

In [24]:
Ciliated_ai = ancestral_input_data(Ciliated,'Ciliated')
Ciliated_ai

,Psp_I.Ciliated,Pan_I.Ciliated,Pse_I.Ciliated,Lre_G.Ciliated,Mat_S.Ciliated,Psp_S.Ciliated,Pan_D.Ciliated,Pse_S.Ciliated,Cpl_S.Ciliated,Asp_S.Ciliated,id,counts
0,1,1,1,1,1,1,1,1,1,1,OrthoGroup9944,10 (Ciliated)
1,1,1,1,1,1,1,1,1,1,1,OrthoGroup11905,10 (Ciliated)
2,1,1,1,1,1,1,1,1,1,1,OrthoGroup5297,10 (Ciliated)
3,1,1,1,1,1,1,1,1,1,1,OrthoGroup4968,10 (Ciliated)
4,1,1,1,1,1,1,1,1,1,1,OrthoGroup7847,10 (Ciliated)
...,...,...,...,...,...,...,...,...,...,...,...,...
504,1,0,1,0,0,1,0,0,0,0,OrthoGroup2449,3 (Ciliated)
505,1,0,0,0,0,1,0,0,0,1,OrthoGroup5091,3 (Ciliated)
506,1,0,1,0,0,1,0,0,0,0,OrthoGroup6594,3 (Ciliated)
507,0,0,1,0,1,0,0,0,0,1,OrthoGroup12570,3 (Ciliated)


In [25]:
#Tuft

Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[10])
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[22])
Psp_I = find_specfic_cluster(csv_data_dic['Psp.I'],[22])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[5])#？
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[23])
Cpl_I = find_specfic_cluster(csv_data_dic['Cpl.I'],[11,14])
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[22]) #？
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[6])
Hsa_I = find_specfic_cluster(csv_data_dic['Hsa.I'],[27,33])#？

Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[21])
Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[24,32])
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[12])
Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[11])
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[29])
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[12,18,10])

Tuft = {
    'Ome_I.Tuft':Ome_I,
    'Mat_I.Tuft':Mat_I,
    'Psp_I.Tuft':Psp_I,
    'Pan_I.Tuft':Pan_I,
    'Pse_I.Tuft':Pse_I,
    'Cpl_I.Tuft':Cpl_I,
    'Pbu_I.Tuft':Pbu_I,
    'Asp_I.Tuft':Asp_I,
    'Hsa_I.Tuft':Hsa_I,
    
    'Ome_D.Tuft':Ome_S,
    'Mat_S.Tuft':Mat_S,
    'Psp_S.Tuft':Psp_S,
    'Pse_S.Tuft':Pse_S,
    'Pbu_S.Tuft':Pbu_S,
    'Asp_S.Tuft':Asp_S, 
}

In [26]:
Tuft_ai = ancestral_input_data(Tuft,'Tuft')
Tuft_ai

,Ome_I.Tuft,Mat_I.Tuft,Psp_I.Tuft,Pan_I.Tuft,Pse_I.Tuft,Cpl_I.Tuft,Pbu_I.Tuft,Asp_I.Tuft,Hsa_I.Tuft,Ome_D.Tuft,Mat_S.Tuft,Psp_S.Tuft,Pse_S.Tuft,Pbu_S.Tuft,Asp_S.Tuft,id,counts
0,1,1,0,1,1,1,0,1,0,1,1,0,1,1,1,OrthoGroup4559,11 (Tuft)
1,1,1,0,0,1,1,0,1,0,1,1,1,1,0,1,OrthoGroup10226,10 (Tuft)
2,1,1,0,0,1,0,0,0,0,1,1,1,1,1,1,OrthoGroup3575,9 (Tuft)
3,1,1,0,0,1,0,0,1,0,0,1,1,1,0,1,OrthoGroup11720,8 (Tuft)
4,0,1,0,0,1,1,0,1,0,0,1,0,1,1,1,OrthoGroup10232,8 (Tuft)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,RNF225,4 (Tuft)
82,0,0,0,0,0,0,0,1,0,0,1,0,0,1,1,OrthoGroup2225,4 (Tuft)
83,0,0,0,1,0,1,0,1,0,0,0,0,0,0,1,OrthoGroup2852,4 (Tuft)
84,0,0,0,1,0,1,0,1,0,0,0,0,0,0,1,OrthoGroup3364,4 (Tuft)


In [27]:
#BEST4
Ome_I = find_specfic_cluster(csv_data_dic['Ome.I'],[13])
Mat_I = find_specfic_cluster(csv_data_dic['Mat.I'],[9])
Pan_I = find_specfic_cluster(csv_data_dic['Pan.I'],[13,15])
Pse_I = find_specfic_cluster(csv_data_dic['Pse.I'],[19])
Pbu_I = find_specfic_cluster(csv_data_dic['Pbu.I'],[18,10])
Asp_I = find_specfic_cluster(csv_data_dic['Asp.I'],[17])
Hsa_I = find_specfic_cluster(csv_data_dic['Hsa.I'],[19])

Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[35])
Hsa_S = find_specfic_cluster(csv_data_dic['Hsa.S'],[10])#？

BEST4 = {
    'Ome_I.BEST4':Ome_I,
    'Mat_I.BEST4_like':Mat_I,
    'Pan_I.BEST4':Pan_I,
    'Pse_I.BEST4':Pse_I,
    'Pbu_I.BEST4':Pbu_I,
    'Asp_I.BEST4':Asp_I,
    'Hsa_I.BEST4':Hsa_I,
    
    'Ome_D.BEST4':Ome_S,
    'Hsa_S.BEST4':Hsa_S,
}

In [28]:
BEST4_ai = ancestral_input_data(BEST4,'BEST4')
BEST4_ai

,Ome_I.BEST4,Mat_I.BEST4_like,Pan_I.BEST4,Pse_I.BEST4,Pbu_I.BEST4,Asp_I.BEST4,Hsa_I.BEST4,Ome_D.BEST4,Hsa_S.BEST4,id,counts
0,0,0,0,1,1,1,1,0,1,OrthoGroup6400,5 (BEST4)
1,1,1,0,0,0,0,1,0,1,OrthoGroup5635,4 (BEST4)
2,1,0,0,1,0,1,0,1,0,OrthoGroup1383,4 (BEST4)
3,1,1,1,0,0,0,0,0,1,OrthoGroup2380,4 (BEST4)
4,0,0,0,0,1,1,1,0,1,OrthoGroup11090,4 (BEST4)
...,...,...,...,...,...,...,...,...,...,...,...
56,0,0,0,1,0,0,0,1,1,OrthoGroup8421,3 (BEST4)
57,0,0,0,1,0,0,1,0,1,OrthoGroup3934,3 (BEST4)
58,1,0,0,0,0,0,1,0,1,MYO6,3 (BEST4)
59,1,1,0,0,0,0,0,0,1,OrthoGroup584,3 (BEST4)


In [29]:
#Oxynticopeptic
# # Ome_S = find_specfic_cluster(csv_data_dic['Ome.S'],[])
# Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[13,1,12,19])
# Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[14,17])
# # Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[])
# Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[13,26,20])
# Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[5])
# Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[28])
# Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[8,1,16])
# Hsa_S = find_specfic_cluster(csv_data_dic['Hsa.S'],[24,3,21,10])

Mat_S = find_specfic_cluster(csv_data_dic['Mat.S'],[11,30])
Psp_S = find_specfic_cluster(csv_data_dic['Psp.S'],[14,17])
# Pan_S = find_specfic_cluster(csv_data_dic['Pan.S'],[])
Pse_S = find_specfic_cluster(csv_data_dic['Pse.S'],[13])
Pse_S_like = find_specfic_cluster(csv_data_dic['Pse.S'],[13])
Cpl_S = find_specfic_cluster(csv_data_dic['Cpl.S'],[5])
Pbu_S = find_specfic_cluster(csv_data_dic['Pbu.S'],[28])
Asp_S = find_specfic_cluster(csv_data_dic['Asp.S'],[1,16])
Hsa_S_C = find_specfic_cluster(csv_data_dic['Hsa.S'],[21])
Hsa_S_P = find_specfic_cluster(csv_data_dic['Hsa.S'],[24])
Hsa_S_MN = find_specfic_cluster(csv_data_dic['Hsa.S'],[21,10])#？
Hsa_I_MN = find_specfic_cluster(csv_data_dic['Hsa.S'],[34,33])#？

Oxynticopeptic = {   
    # 'Ome_S':Ome_S,
    'Mat_S.Oxynticopeptic':Mat_S,
    'Psp_S.Oxynticopeptic':Psp_S,
    # 'Pan_S':Pan_S,
    'Pse_S.Oxynticopeptic':Pse_S,
    'Pse_S.Oxynticopeptic_like':Pse_S_like,
    'Cpl_S.Oxynticopeptic':Cpl_S,
    'Pbu_S.Oxynticopeptic':Pbu_S,
    'Asp_S.Oxynticopeptic':Asp_S, 
    'Hsa_S.Chief':Hsa_S_C,
    'Hsa_S.Parietal':Hsa_S_P,
    'Hsa_S.Mucous':Hsa_S_MN,
    'Hsa_I.Mucous':Hsa_I_MN,
}

In [30]:
Oxynticopeptic_ai = ancestral_input_data(Oxynticopeptic,'Oxynticopeptic')
Oxynticopeptic_ai

,Mat_S.Oxynticopeptic,Psp_S.Oxynticopeptic,Pse_S.Oxynticopeptic,Pse_S.Oxynticopeptic_like,Cpl_S.Oxynticopeptic,Pbu_S.Oxynticopeptic,Asp_S.Oxynticopeptic,Hsa_S.Chief,Hsa_S.Parietal,Hsa_S.Mucous,Hsa_I.Mucous,id,counts
0,1,1,1,1,1,1,1,0,1,0,0,OrthoGroup2749,8 (Oxynticopeptic)
1,1,1,1,1,1,0,1,1,0,1,0,OrthoGroup4773,8 (Oxynticopeptic)
2,1,1,1,1,1,1,1,0,1,0,0,OrthoGroup1768,8 (Oxynticopeptic)
3,0,1,1,1,1,1,1,1,0,1,0,OrthoGroup353,8 (Oxynticopeptic)
4,1,0,1,1,1,1,0,1,0,1,0,OrthoGroup8574,7 (Oxynticopeptic)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,1,0,0,0,1,0,1,0,0,0,0,OrthoGroup10738,3 (Oxynticopeptic)
234,1,0,0,0,1,0,1,0,0,0,0,OrthoGroup7368,3 (Oxynticopeptic)
235,1,0,1,1,0,0,0,0,0,0,0,OrthoGroup8323,3 (Oxynticopeptic)
236,1,0,0,0,1,0,1,0,0,0,0,OrthoGroup6321,3 (Oxynticopeptic)


In [31]:
import pandas as pd
import numpy as np

def replace_nan_with_int_zero(df):

    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    
    # 将数值列中的 NaN 替换为 0
    df[numeric_cols] = df[numeric_cols].fillna(0)
    
    # 将所有数值列转换为整数类型
    for col in numeric_cols:
        # 检查列是否包含浮点数（可能因NaN转换产生）
        if pd.api.types.is_float_dtype(df[col]):
            # 确保所有值都是整数（或可以安全转换为整数）
            if (df[col] % 1 == 0).all():
                df[col] = df[col].astype(int)
    
    return df

In [32]:
df_list = [Endocrine_ai,Enterocyte_ai,Endothelial_ai,Mucus_Goblet_ai,Myeloid_ai,Mesenchymal_ai,T_cell_ai,B_cell_ai,Oxynticopeptic_ai,Tuft_ai,BEST4_ai,Ciliated_ai]

merged_df = pd.concat(df_list, ignore_index=True)
df = replace_nan_with_int_zero(merged_df)

In [33]:
df

,Ome_I.Enteroendocrine,Mat_I.Enteroendocrine,Psp_I.Enteroendocrine,Pan_I.Enteroendocrine,Pse_I.Enteroendocrine,Cpl_I.Enteroendocrine,Lre_G.Enteroendocrine,Pbu_I.Enteroendocrine,Asp_I.Enteroendocrine,Hsa_I.Enteroendocrine,...,Psp_I.Ciliated,Pan_I.Ciliated,Pse_I.Ciliated,Lre_G.Ciliated,Mat_S.Ciliated,Psp_S.Ciliated,Pan_D.Ciliated,Pse_S.Ciliated,Cpl_S.Ciliated,Asp_S.Ciliated
0,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,1,1,1,1,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,1,1,1,1,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3225,0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,1,0,0,0,0
3226,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,1
3227,0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,1,0,0,0,0
3228,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,1


In [39]:
df_ortho = df[df['id'].str.contains(r'^Ortho', na=False)]
df1 = df_ortho.groupby('id').sum().reset_index()


In [40]:
df1

,id,Ome_I.Enteroendocrine,Mat_I.Enteroendocrine,Psp_I.Enteroendocrine,Pan_I.Enteroendocrine,Pse_I.Enteroendocrine,Cpl_I.Enteroendocrine,Lre_G.Enteroendocrine,Pbu_I.Enteroendocrine,Asp_I.Enteroendocrine,...,Psp_I.Ciliated,Pan_I.Ciliated,Pse_I.Ciliated,Lre_G.Ciliated,Mat_S.Ciliated,Psp_S.Ciliated,Pan_D.Ciliated,Pse_S.Ciliated,Cpl_S.Ciliated,Asp_S.Ciliated
0,OrthoGroup1000,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,OrthoGroup10000,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,OrthoGroup10006,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,OrthoGroup1001,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,OrthoGroup10014,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2796,OrthoGroup9983,0,0,0,0,0,0,0,0,0,...,1,1,0,1,0,1,1,0,1,1
2797,OrthoGroup9993,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2798,OrthoGroup9995,0,0,0,0,0,0,0,0,0,...,0,1,0,1,1,0,1,0,1,1
2799,OrthoGroup9997,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [42]:
df1.to_csv('D:/project/digestion_SI/04.CCM/Allcelltype_test2/All_module_gene_ortho(2026.04.30).0.2.csv',index = False)